# الفصل 35: أساسيات النماذج متعددة الوسائط

> حتى وقت قريب كانت النماذج اللغوية تعيش في عالم من الرموز النصية فقط.

- **الجزء:** 8 — النماذج متعددة الوسائط
- **الفصل:** 35
- **المستوى:** متقدم
- **زمن القراءة:** 15 دقيقة
- **المتطلبات:** الفصل 34: DPO وKTO وORPO: ما بعد RLHF
- **الوسوم:** Multimodal، Modality Gap، Alignment، Any-to-Any

## كيف تشغّل هذا الدفتر؟

1. **Google Colab:** ارفع الملف عبر `File → Upload notebook`، ثم `Runtime → Run all`.
2. **Kaggle:** أنشئ Notebook ثم `File → Import notebook`، ثم `Run All`.
3. شغّل خلية **إعداد البيئة** أولًا؛ ستفحص الحزم وتثبّت الناقص تلقائيًا.
4. للتدريب أو النماذج الكبيرة: Colab `Runtime → Change runtime type → T4 GPU`، وKaggle `Settings → Accelerator → GPU`.


<div dir="rtl" style="margin:2px 0 18px;line-height:2.4;">
<b>افتح هذا الدفتر على منصة مجانية:</b><br>
<a href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-08-multimodal/ch35-multimodal-basics.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#F9AB00;color:#111111;">Google Colab</a> <a href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-08-multimodal%2Fch35-multimodal-basics.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#20BEFF;color:#FFFFFF;">Kaggle</a> <a href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-08-multimodal/ch35-multimodal-basics.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#579ACA;color:#FFFFFF;">Binder</a> <a href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-08-multimodal/ch35-multimodal-basics.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#6C5CE7;color:#FFFFFF;">Gradient</a> <a href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-08-multimodal/ch35-multimodal-basics.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#37474F;color:#FFFFFF;">nbviewer</a> <a href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-08-multimodal/ch35-multimodal-basics.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#24292F;color:#FFFFFF;">GitHub</a>
</div>


## ملاحظات التشغيل

- **الحزم:** torch
- **GPU:** يستخدم هذا الدفتر PyTorch أو نماذج كبيرة؛ يُفضّل GPU. يمكنك البدء بنماذج صغيرة على CPU، لكن التدريب الكامل سيكون بطيئًا.
- **مقاطع توضيحية:** بعض الشيفرة تحتوي `...` أو `NotImplementedError`؛ هذا مقصود لبيان البنية وليس خطأً في الدفتر.


<style>
/* نص عربي من اليمين إلى اليسار، مع إبقاء الشيفرة من اليسار */
.rendered_html, .jp-RenderedHTMLCommon, .text_cell_render, .markdown-body {
  direction: rtl;
  text-align: right;
}
.rendered_html pre, .rendered_html code,
.jp-RenderedHTMLCommon pre, .jp-RenderedHTMLCommon code,
.text_cell_render pre, .text_cell_render code,
.markdown-body pre, .markdown-body code {
  direction: ltr;
  text-align: left;
  unicode-bidi: embed;
}
.rendered_html table, .jp-RenderedHTMLCommon table, .text_cell_render table {
  direction: rtl;
}
</style>


In [ ]:
# === إعداد البيئة ===
# يفحص هذا القسم الحزم المطلوبة ويحاول تثبيت الناقص منها تلقائيًا.
# شغّل هذه الخلية أولًا، ثم بقية خلايا الدفتر.
import importlib.util
import subprocess
import sys

print("Python:", sys.version.split()[0])

# (اسم وحدة الاستيراد، اسم حزمة pip)
REQUIRED = [
    ("torch", "torch")
]
AUTO_INSTALL = {"PIL", "bs4", "datasets", "faiss", "fasttext", "librosa", "networkx", "opentelemetry", "peft", "prometheus_client", "pydantic", "rank_bm25", "requests", "sentence_transformers", "sentencepiece", "soundfile", "tokenizers", "trafilatura", "transformers", "wandb", "yaml"}


def has_module(name):
    try:
        return importlib.util.find_spec(name) is not None
    except Exception:
        return False


if not REQUIRED:
    print("لا توجد مكتبات خارجية مطلوبة؛ يكفي Python القياسي.")
else:
    missing = [(module, package) for module, package in REQUIRED if not has_module(module)]
    auto_specs = []
    manual_specs = []
    for module, package in missing:
        specs = package.split()
        if module in AUTO_INSTALL:
            auto_specs.extend(specs)
        else:
            manual_specs.extend(specs)
    auto_specs = list(dict.fromkeys(auto_specs))
    manual_specs = list(dict.fromkeys(manual_specs))

    if auto_specs:
        print("⏳ تثبيت تلقائي:", " ".join(auto_specs))
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *auto_specs])
            print("✅ تم التثبيت.")
        except Exception as exc:
            print("⚠️ تعذّر التثبيت التلقائي:", exc)
            print("شغّل يدويًا: %pip install", " ".join(auto_specs))
    if manual_specs:
        print("⚠️ حزم تحتاج تثبيتًا يدويًا:", " ".join(manual_specs))
        print("شغّل: %pip install", " ".join(manual_specs))
    if not auto_specs and not manual_specs:
        print("✅ كل الحزم المطلوبة متاحة.")

    print("\nالحالة النهائية:")
    for module, package in REQUIRED:
        print(("✅" if has_module(module) else "⚠️"), package)


<div class="notebook-actions" dir="rtl"><span class="notebook-actions__label">شغّل الدفتر:</span><a class="nb-action nb-action--colab" href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-08-multimodal/ch35-multimodal-basics.ipynb" target="_blank" rel="noopener" title="افتح في Google Colab — GPU مجاني">Google Colab</a><a class="nb-action nb-action--kaggle" href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-08-multimodal%2Fch35-multimodal-basics.ipynb" target="_blank" rel="noopener" title="افتح في Kaggle — GPU/TPU مجاني">Kaggle</a><a class="nb-action nb-action--binder" href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-08-multimodal/ch35-multimodal-basics.ipynb" target="_blank" rel="noopener" title="افتح في Binder — بيئة Jupyter مجانية">Binder</a><a class="nb-action nb-action--gradient" href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-08-multimodal/ch35-multimodal-basics.ipynb" target="_blank" rel="noopener" title="افتح في Paperspace Gradient — GPU مجاني محدود">Gradient</a><a class="nb-action nb-action--nbviewer" href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-08-multimodal/ch35-multimodal-basics.ipynb" target="_blank" rel="noopener" title="اعرض الدفتر دون تنفيذ">nbviewer</a><a class="nb-action nb-action--github" href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-08-multimodal/ch35-multimodal-basics.ipynb" target="_blank" rel="noopener" title="اعرض الملف المصدر على GitHub">GitHub</a></div>

> **ملخص الفصل**
>
> حتى وقت قريب كانت النماذج اللغوية تعيش في عالم من الرموز النصية فقط.

> **بيانات الفصل**
>
> - **الجزء:** 8 — النماذج متعددة الوسائط
> - **المستوى:** متقدم
> - **زمن القراءة:** 15 دقيقة
> - **المتطلبات:** الفصل 34: DPO وKTO وORPO: ما بعد RLHF


## مخرجات التعلم

بنهاية هذا الفصل ستكون قادرًا على:

- فهم فجوة الوسيط (Modality Gap)
- فهم من البكسل إلى الرمز
- فهم استراتيجيات المحاذاة والدمج


## 1. لماذا يهمّك هذا الموضوع؟

حتى وقت قريب كانت النماذج اللغوية تعيش في عالم من الرموز النصية فقط. لكن معظم
المعلومات التي ينتجها البشر ليست نصًا: الصور، الصوت، الفيديو، الإشارات الطبية،
والخرائط. عندما نطلب من نموذج أن «يفهم» صورة أو «يسمع» تسجيلًا، فإننا نطلب منه
أن يعبر فجوة تمثيلية عميقة: النص منفصل (Discrete) ومحدود القاموس، بينما الصوت
والصورة مستمران (Continuous) وعاليَا الأبعاد.

هذه الفجوة ليست تفصيلًا تقنيًا؛ إنها تحدد كل قرار في تصميم النموذج:

- كيف نحوّل صورة 224×224×3 إلى تسلسل يمكن لمحوّل أن يعالجه؟
- كيف نجعل تمثيل الصورة «قريبًا» من تمثيل النص الذي يصفها؟
- كيف نمنع الصورة من استهلاك آلاف الرموز وإغراق سياق النموذج؟
- كيف ندرّب النموذج عندما تكون أزواج (صورة، نص) وفيرة لكن أزواج (سؤال، جواب) نادرة؟

الإجابة العملية اليوم هي **النماذج متعددة الوسائط (Multimodal Models)**: نماذج
تدمج مُشفِّرات متخصصة لكل وسيط مع نموذج لغوي مركزي. هذا الفصل يبني الخريطة
المفاهيمية التي تعتمد عليها الفصول الستة التالية: ViT وCLIP وVLMs والانتشار
والصوت والنماذج الموحّدة.


## 2. الخريطة الذهنية


```text
النموذج متعدد الوسائط
│
├── المُدخلات (Inputs)
│   ├── نص    → Tokenizer → Embeddings
│   ├── صورة  → Patchification → Vision Encoder → Visual Tokens
│   ├── صوت   → Spectrogram / Waveform → Audio Encoder → Audio Tokens
│   └── فيديو → إطارات + زمن → Video Encoder → Spatiotemporal Tokens
│
├── الجسر (Alignment / Fusion)
│   ├── Contrastive (CLIP)          فضاء مشترك للمقارنة
│   ├── Projection (LLaVA)          إسقاط بصري إلى فضاء LLM
│   ├── Cross-Attention (Flamingo)  انتباه بين النص والصورة
│   └── Unified Tokens (Chameleon)  كل الوسائط في قاموس واحد
│
└── المُخرجات (Outputs)
    ├── نص، إجابة، وصف
    ├── صورة (Diffusion)
    ├── صوت (TTS)
    └── إجراء (Agent)
```


السؤال المعماري الأول: **هل نُحاذي التمثيلات في فضاء مشترك للمقارنة فقط، أم
نُدخل الوسائط فعليًا إلى النموذج اللغوي؟** الإجابة تحدد ما إذا كان النموذج
«مُشفِّرًا مزدوجًا» (Dual Encoder) مثل CLIP، أم «نموذج رؤية-لغة» (VLM) مثل
LLaVA، أم «نموذجًا موحّدًا» (Any-to-Any) مثل GPT-4o وGemini.


## 3. الشرح المتدرج

### 3.1 فجوة الوسيط (Modality Gap)

كل وسيط له بنية إحصائية مختلفة:

| الوسيط | التمثيل الخام | الطبيعة | التحدي |
|---|---|---|---|
| نص | تسلسل رموز | منفصل، رمزي | مفردات كبيرة، سياق |
| صورة | مصفوفة بكسلات | مستمر، مكاني | أبعاد ضخمة، محلية |
| صوت | موجة/طيف | مستمر، زمني | ترددات، ضوضاء |
| فيديو | إطارات + زمن | مستمر، مكاني-زمني | حجم هائل، ازدواجية |

حتى لو كان لبُعد التضمين نفس الحجم (مثلًا 768)، فإن توزيع متجهات الصور
ومتجهات النص يختلفان. هذا هو **Modality Gap**: المسافة بين مراكز التوزيعين
في الفضاء المشترك. المحاذاة (Alignment) هي عملية تقليل هذه الفجوة بحيث تصبح
المقارنة بين «صورة قطة» و«نص قطة» ذات معنى.

### 3.2 من البكسل إلى الرمز

أكثر طريقة شيوعًا لتحويل الصورة إلى تسلسل هي **التقطيع إلى رقع (Patchification)**:

1. تُقسَّم الصورة إلى رقع مربعة غير متداخلة (مثلًا 16×16 بكسل).
2. كل رقعة تُسطَّح إلى متجه (16×16×3 = 768 قيمة).
3. يُسقَط المتجه خطيًا إلى بُعد النموذج (`d_model`).
4. تُضاف تضمينات موضعية، ثم تُمرَّر عبر محوّل.

لصورة 224×224 ورقع 16×16 نحصل على:

\[
N_{\text{patches}} = \left(\frac{224}{16}\right)^2 = 14^2 = 196
\]

أي 196 «توكنًا بصريًا» — أكثر من معظم الجمل النصية. وعند رفع الدقة إلى
336×336 تصبح 441 رقعة، وإلى 448×448 تصبح 784. هذا يفسّر لماذا يصبح طول
السياق وكلفة الانتباه تحديًا مركزيًا في VLMs.

### 3.3 استراتيجيات المحاذاة والدمج

**أ) الترميز المزدوج التبايني (CLIP / SigLIP).**
مُشفِّران منفصلان (نص وصورة) يُدرَّبان معًا على أزواج (صورة، وصف). الهدف:
تعظيم تشابه الزوج الصحيح وتقليل تشابه الأزواج الخاطئة في الدفعة. الناتج فضاء
مشترك يسمح بالبحث والتصنيف دون توليد نص.

**ب) الإسقاط إلى فضاء LLM (LLaVA).**
نُبقي مُشفِّر الرؤية مجمّدًا، ونُدرّب طبقة إسقاط (Linear أو MLP) تحوّل
تمثيلات الرقع إلى نفس بُعد تضمين النص. ثم نُدخل الرموز البصرية في تسلسل
المدخلات كما لو كانت كلمات. هذا هو أبسط تصميم VLM وأكثرها شيوعًا.

**ج) الانتباه المتقاطع (Flamingo).**
بدل حقن الرموز البصرية في البداية، نُضيف طبقات Cross-Attention بين طبقات
الـLLM، فتستطيع كل طبقة نصية «الانتباه» إلى تمثيلات الصورة. هذا يدعم تعدد
الصور بمرونة أكبر، لكنه أكثر تعقيدًا في التنفيذ والتدريب.

**د) التوحيد الكامل (Chameleon / GPT-4o).**
نُحوّل كل الوسائط (نص، صورة، صوت) إلى رموز منفصلة في قاموس موحّد، ثم نُدرّب
محوّلًا واحدًا على التنبؤ بالرمز التالي. هذا هو الطموح النهائي لـ«Any-to-Any»،
لكنه يتطلب توفير مُرمِّزات (Tokenizers) عالية الجودة لكل وسيط وتوازنًا دقيقًا
في خلط البيانات.

### 3.4 متى نُجمّد ومتى نُدرّب؟

القاعدة العملية في معظم VLMs الحديثة:

- **مُشفِّر الرؤية:** مجمّد في البداية (لأنه مُدرَّب على مليارات أزواج).
- **طبقة الإسقاط:** تُدرَّب دائمًا (هي الجسر).
- **الـLLM:** مجمّد في مرحلة المحاذاة الأولى، ثم يُفتح جزئيًا (LoRA) في مرحلة
  التعليمات البصرية.

هذا التقسيم يقلل التكلفة ويحمي التمثيلات المُدرَّبة مسبقًا من النسيان الكارثي.

### 3.5 التحديات الحسابية

| التحدي | السبب | التخفيف |
|---|---|---|
| طول السياق | مئات الرموز البصرية لكل صورة | تقليل الرقع، Perceiver Resampler، ضغط الرموز |
| كلفة الانتباه | \(O(n^2)\) مع تسلسل طويل | Flash Attention، سياق منزلق |
| خلط البيانات | نسب الوسائط غير متوازنة | أخذ عينات موزون، مراقبة النسب |
| الهلوسة البصرية | النموذج يصف ما لا يراه | تقييم POPE، بيانات سلبية، RLHF بصري |
| التقييم | لا مقياس واحد يكفي | MMMU، MMBench، ChartQA، تقييم بشري |


## 4. مثال محلول خطوة بخطوة

**الهدف:** حساب مصفوفة التشابه بين تمثيلين لصورتين وتمثيلين لنصين، ثم قراءة
أي زوج «صحيح».


In [ ]:
import torch
import torch.nn.functional as F

def cosine_similarity_matrix(image_emb: torch.Tensor,
                             text_emb: torch.Tensor) -> torch.Tensor:
    """يعيد مصفوفة [num_images, num_texts] لتشابه جيب التمام."""
    image_emb = F.normalize(image_emb, dim=-1)
    text_emb = F.normalize(text_emb, dim=-1)
    return image_emb @ text_emb.T

# تمثيلان بصريان (بُعد 3 للتوضيح) وتمثيلان نصيان
image_emb = torch.tensor([
    [1.00, 0.20, 0.00],   # صورة قطة
    [0.10, 1.00, 0.10],   # صورة سيارة
])
text_emb = torch.tensor([
    [0.90, 0.30, 0.10],   # "قطة"
    [0.20, 0.80, 0.20],   # "سيارة"
])

sim = cosine_similarity_matrix(image_emb, text_emb)
print(sim)
# المتوقع:
# tensor([[0.98, 0.43],
#         [0.38, 0.97]])


**القراءة:** القطر الرئيسي (0.98 و0.97) هو تشابه الأزواج الصحيحة، والقيم
الخارجية (0.43 و0.38) أقل بكثير. هذا هو الهدف الذي يُدرَّب عليه CLIP: تكبير
القطر وتقليل الباقي. لاحظ أن القيم ليست 1 تمامًا لأن التمثيلات ليست متطابقة
اتجاهيًا؛ المحاذاة الجيدة تجعل القطر أعلى من أي زوج خاطئ بهامش واضح.

**خطوة إضافية:** جرّب إضافة ضوضاء صغيرة إلى `image_emb` وأعد الحساب. ستلاحظ
أن الترتيب قد ينقلب إذا كانت المسافة بين الأزواج صغيرة — وهذا يوضّح لماذا
يحتاج النموذج إلى هامش (Margin) ودرجة حرارة (Temperature) في التدريب.


## 5. نصائح وحيل

- **ابدأ من مُشفِّر مُدرَّب مسبقًا** (CLIP/SigLIP) ولا تُدرّب ViT من الصفر إلا إذا كانت لديك بيانات ضخمة.
- **جمّد مُشفِّر الرؤية في المرحلة الأولى**، ثم افتحه جزئيًا فقط إذا لزم الأمر.
- **اضبط عدد الرموز البصرية** بعناية: 196 رمزًا قد تكون كثيرة لمهمة تصنيف، وقليلة لمهمة OCR.
- **استخدم Perceiver Resampler أو Q-Former** عندما تحتاج عددًا ثابتًا وصغيرًا من الرموز.
- **وازن نسب الوسائط** في التدريب؛ لا تدع النص يطغى على الصورة أو العكس.
- **قيّم الهلوسة البصرية** بشكل منفصل عن الدقة العامة (POPE، أسئلة سلبية).
- **اختبر على دقة صور مختلفة**؛ كثير من النماذج تضعف خارج دقة التدريب.
- **وثّق preprocessing** (تطبيع، resize، crop) لأنه جزء من النموذج فعليًا.


## 6. أخطاء شائعة ومصائد

- **الاعتقاد أن الصورة «تُدخل» كما هي إلى LLM.** لا بد من مُشفِّر وتحويل إلى تمثيلات.
- **تجاهل فجوة الوسيط.** مجرد تطابق الأبعاد لا يعني أن التمثيلات قابلة للمقارنة.
- **استخدام عدد رقع ثابت دون مراعاة الدقة.** صورة عالية الدقة قد تفقد تفاصيل حرجة عند تصغيرها.
- **تدريب طبقة الإسقاط وحدها إلى الأبد.** المرحلة الثانية (تعليمات بصرية) ضرورية للأداء الحقيقي.
- **إهمال البيانات السلبية.** بدون أمثلة «صورة لا تطابق النص»، يتعلم النموذج الربط السطحي.
- **تقييم النموذج على نص فقط.** VLM قد يكون ممتازًا في اللغة وضعيفًا في الإدراك البصري.
- **نسيان تكلفة الاستدلال.** كل صورة تضيف مئات الرموز إلى السياق، ما يضاعف الكلفة.
- **خلط مقاييس التقييم.** MMMU تقيس المعرفة، POPE تقيس الهلوسة، MMBench تقيس القدرات — لا تستبدل إحداها بالأخرى.


## 7. جدول مقارنة

| النمط | مثال | ما يُدرَّب | قوة | ضعف | الاستخدام |
|---|---|---|---|---|---|
| Dual Encoder | CLIP، SigLIP | المُشفِّران + الإسقاط | بحث/تصنيف سريع، فضاء مشترك | لا يولّد نصًا حرًا | Retrieval، zero-shot classification |
| Projection VLM | LLaVA، Qwen-VL | الإسقاط (+LLM لاحقًا) | بسيط، قوي، شائع | رموز بصرية كثيرة | مساعد بصري عام |
| Cross-Attention VLM | Flamingo، IDEFICS | طبقات cross-attention | تعدد صور، مرونة | أعقد، أغلى | few-shot بصري |
| Unified Any-to-Any | Chameleon، GPT-4o | كل شيء | توليد وفهم موحّد | بيانات وتوكنزة معقدة | مساعد متعدد الوسائط كامل |


## 8. ملخص أفضل الممارسات

- **المعمارية تتبع المهمة:** CLIP للبحث، LLaVA للمساعد البصري، Flamingo لتعدد الصور، Unified للمنتج الشامل.
- **المحاذاة قبل التعليمات:** درّب الجسر أولًا، ثم علّم النموذج اتباع التعليمات.
- **التجميد الذكي:** جمّد ما هو مُدرَّب جيدًا، درّب ما هو جديد.
- **قلّل الرموز:** استخدم ضغط الرموز أو resampler عند الحاجة.
- **قيّم على محاور متعددة:** فهم، هلوسة، OCR، استدلال، عدالة.
- **راقب التكلفة:** عدد الرموز البصرية يحدد زمن الاستجابة والتكلفة.
- **وثّق كل شيء:** preprocessing، نسب البيانات، إصدارات المُشفِّرات.


## 9. أسئلة مقابلات

**س: لماذا لا يمكن حقن بكسلات الصورة الخام مباشرة في محوّل نصي؟**

ج: لأن المحوّل يتوقع متجهات تضمين بأبعاد ومعنى دلالي متوافق مع فضاء الكلمات. البكسلات الخام عالية الأبعاد وغير مهيكلة دلاليًا (224×224×3 = 150,528 قيمة)، ولا تحمل تمثيلًا مجردًا للأشياء. مُشفِّر الرؤية يحوّلها إلى تمثيلات مكثفة ودلالية قابلة للمقارنة مع النص.

**س: ما الفرق بين Dual Encoder وFusion VLM؟**

ج: Dual Encoder (CLIP) يُنتج تمثيلين منفصلين للمقارنة فقط، ولا يولّد نصًا. Fusion VLM (LLaVA) يُدخل تمثيلات الصورة في تسلسل LLM، فيستطيع توليد إجابة نصية حرة معقدة. الأول مناسب للاسترجاع والتصنيف، الثاني للمساعدات الحوارية.

**س: كيف تتعامل مع صور عالية الدقة في VLM؟**

ج: عبر تقليل عدد الرقع (patch size أكبر)، أو تقسيم الصورة إلى tiles ومعالجة كل tile، أو استخدام resampler يضغط الرموز إلى عدد ثابت. الحل الأمثل يعتمد على المهمة: OCR يحتاج دقة عالية، بينما التصنيف العام لا يحتاجها.

**س: لماذا تُجمَّد مُشفِّرات الرؤية عادة؟**

ج: لأنها مُدرَّبة على مليارات أزواج (صورة، نص) وتمتلك تمثيلًا عامًا قويًا. إعادة تدريبها على بيانات تعليمات أصغر قد تُدمّر هذا التمثيل (Catastrophic Forgetting) بتكلفة حسابية كبيرة مقابل مكسب محدود.


## 10. تمرين عملي

1. حمّل `openai/clip-vit-base-patch32` من Hugging Face.
2. احسب تمثيلات 5 صور و5 نصوص إنجليزية (أو عربية مع نموذج متعدد اللغات).
3. احسب مصفوفة التشابه `5×5` وحدد الزوج الصحيح لكل صورة.
4. أضف ضوضاء Gaussian بمستويات مختلفة (0.01، 0.05، 0.1) إلى تمثيلات الصور، وأعد حساب دقة المطابقة.
5. ارسم منحنى «الدقة مقابل شدة الضوضاء» وفسّر النتيجة.
6. **(تحدٍّ إضافي):** كرر التجربة مع صور عربية (لافتات/مستندات) وقارن الأداء. هل يضعف النموذج؟ لماذا؟


## 11. مراجع للتعمّق

- Radford et al. (2021): *Learning Transferable Visual Models From Natural Language Supervision* (CLIP).
- Jia et al. (2021): *Scaling Up Visual and Vision-Language Representation Learning With Noisy Text Supervision* (ALIGN).
- Alayrac et al. (2022): *Flamingo: a Visual Language Model for Few-Shot Learning*.
- Li et al. (2023): *BLIP-2: Bootstrapping Language-Image Pre-training with Frozen Image Encoders and Large Language Models*.
- Liu et al. (2023): *Visual Instruction Tuning* (LLaVA).
- Yin et al. (2024): *A Survey on Multimodal Large Language Models*.
- Chameleon Team (2024): *Chameleon: Mixed-Modal Early-Fusion Foundation Models*.


## 🧪 مساحة تجربة

استخدم الخلية التالية لتجربة أفكار الفصل أو حلّ التمرين. عدّل القيم وأعد تشغيل الخلية.


In [ ]:
# 🧪 مساحة تجربة
# اكتب هنا حلّ التمرين أو أي تجربة إضافية.


---
📘 **المصدر:** كتاب *إتقان المحوّلات والنماذج اللغوية الكبيرة* — النماذج متعددة الوسائط / الفصل 35: أساسيات النماذج متعددة الوسائط.

هذا الدفتر مولّد آليًا من ملف الفصل؛ لا تعدّله يدويًا، بل عدّل المصدر ثم أعد البناء.
